# Causal Inference ~ Group 10

## Part A ~ Reduced Form Regression

In [1]:
%conda install -c conda-forge linearmodels

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import statsmodels.api as sm

# Load data (adjust path if needed)
df = pd.read_csv("causal_data/ae_d.csv")

# Define variables
Y = df["INCOME1M"]
Z = df["same_s"]

# Add constant
X = sm.add_constant(Z)

# Run regression
model = sm.OLS(Y, X, missing='drop').fit()

# Print results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               INCOME1M   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     10.07
Date:                Mon, 13 Apr 2026   Prob (F-statistic):            0.00151
Time:                        17:07:19   Log-Likelihood:            -6.5548e+06
No. Observations:              655169   AIC:                         1.311e+07
Df Residuals:                  655167   BIC:                         1.311e+07
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3550.4189      9.402    377.606      0.0

## Part A Interpretation
The reduced-form estimate shows that mothers whose first two children are the same sex earn about $42 less annually, on average, than those with mixed-sex children, and this effect is statistically significant. However, the magnitude is extremely small and the R^2 is basically 0. This means that sex composition explains almost none of the variation in income. This suggests that while the instrument has a statistically detectable effect on income, it might be economically negligible.

## Part B ~ IV Regression

In [3]:
from linearmodels.iv import IV2SLS

# Drop missing values for relevant variables
df_iv = df[["INCOME1M", "more_2", "same_s"]].dropna()

# Run IV regression
iv_model = IV2SLS.from_formula(
    "INCOME1M ~ 1 + [more_2 ~ same_s]",
    data=df_iv
).fit()

# Print results
print(iv_model.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:               INCOME1M   R-squared:                      0.0077
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0077
No. Observations:              655169   F-statistic:                    10.144
Date:                Mon, Apr 13 2026   P-value (F-stat)                0.0014
Time:                        17:07:20   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      3823.2     92.512     41.327     0.0000      3641.9      4004.5
more_2        -766.88     240.78    -3.1849     0.00

## Part B Interpretation
The IV estimate implies that, for compliers, having more than two children reduces maternal income by about $767 per year; this effect is statistically significant. Under the LATE framework, this captures the causal effect of an additional child for women whose fertility decisions are influenced by having two children of the same sex. This suggests a meaningful negative impact of additional children on mothers’ earnings.

## Part C ~ Difference in Part A and B Estimates
The main difference is that OLS and reduced form is not isolating causal variation while the IV regression is. The reduced form only captures the total, weak, effect of the instrument on income, which is small because the instrument minimally affects fertility. On the other hand, the estimate produced by the IV regression rescales this small reudced-form effect by the first stage, isolating the causal effect of having more children. To summarize, IV zooms in on causal effect for a specific group while OLS averages over everyone and is biased by selection for those women who choose larger families.

## Part D ~ Wald Calculation

In [4]:
import numpy as np

# Drop missing values
df_wald = df[["INCOME1M", "more_2", "same_s"]].dropna()

# Compute covariances
cov_yz = np.cov(df_wald["INCOME1M"], df_wald["same_s"], bias=True)[0, 1]
cov_dz = np.cov(df_wald["more_2"], df_wald["same_s"], bias=True)[0, 1]

# Wald estimate
wald_estimate = cov_yz / cov_dz

print("Wald IV estimate:", wald_estimate)


Wald IV estimate: -766.8814148272486


## Part E ~ Repeat for Paternal Income

In [20]:
print("Reduced form for paternal income:\n\n")

df_men = df[["INCOME1D", "same_s"]].dropna()

Y_m = df_men["INCOME1D"]
Z_m = df_men["same_s"]

X_m = sm.add_constant(Z_m)

model_m = sm.OLS(Y_m, X_m).fit()

print(model_m.summary())

df_iv_m = df[["INCOME1D", "more_2", "same_s"]].dropna()

iv_model_m = IV2SLS.from_formula(
    "INCOME1D ~ 1 + [more_2 ~ same_s]",
    data=df_iv_m
).fit()
print()
print("\n\nIV regression for paternal:\n\n")

print(iv_model_m.summary)

Reduced form for paternal income:


                            OLS Regression Results                            
Dep. Variable:               INCOME1D   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                  0.009706
Date:                Mon, 13 Apr 2026   Prob (F-statistic):              0.922
Time:                        17:22:25   Log-Likelihood:            -6.0866e+06
No. Observations:              561459   AIC:                         1.217e+07
Df Residuals:                  561457   BIC:                         1.217e+07
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        1.7

## Part E Interpretation
The results/output suggest that having more than two children substantially reduces mothers’ income but has essentially no effect on fathers’ income. The paternal estimates are near zero and statistically insignificant, indicating that additional children do not meaningfully affect men’s earnings. This implies that increased family size creates a division of labor between the man and woman where woman take on the cost of additional children. As a result, having more children contributes to a widening of the gender wage gap among couples with at least two children in 1980.